# Filters

> Filter implementations for file discovery.

In [ ]:
#| default_exp scanning.filters

In [ ]:
#| export
import fnmatch
from typing import Callable, List, Optional

from cjm_file_discovery.core.models import FileInfo, FileType
from cjm_file_discovery.core.config import FilterConfig

## Filter Functions

Helper functions for filtering file lists.

In [ ]:
#| export
def filter_files(
    files: List[FileInfo],      # Files to filter
    config: FilterConfig        # Filter configuration
) -> List[FileInfo]:  # Filtered files
    """Filter a list of files using FilterConfig."""
    return [f for f in files if config.matches(f)]

In [ ]:
# Test filter_files
test_files = [
    FileInfo(name="doc.py", path="/doc.py", is_directory=False, extension="py", file_type=FileType.CODE),
    FileInfo(name="song.mp3", path="/song.mp3", is_directory=False, extension="mp3", file_type=FileType.AUDIO),
    FileInfo(name="pic.png", path="/pic.png", is_directory=False, extension="png", file_type=FileType.IMAGE),
    FileInfo(name=".hidden", path="/.hidden", is_directory=False),
]

# Filter by extension
py_filter = FilterConfig(extensions=['py'])
result = filter_files(test_files, py_filter)
assert len(result) == 1
assert result[0].name == "doc.py"

# Filter by file type
audio_filter = FilterConfig(file_types=[FileType.AUDIO])
result = filter_files(test_files, audio_filter)
assert len(result) == 1
assert result[0].name == "song.mp3"

# No filter (but exclude hidden)
no_filter = FilterConfig(include_hidden=False)
result = filter_files(test_files, no_filter)
assert len(result) == 3  # .hidden excluded

print("filter_files tests passed!")

filter_files tests passed!


## Sorting Functions

Helper functions for sorting file lists.

In [ ]:
#| export
def sort_files(
    files: List[FileInfo],      # Files to sort
    sort_by: str = "name",      # Sort key: "name", "size", "modified", "type"
    descending: bool = False    # Sort in descending order
) -> List[FileInfo]:  # Sorted files
    """Sort a list of files by the specified key."""
    def get_sort_key(f: FileInfo):
        if sort_by == "name":
            return f.name.lower()
        elif sort_by == "size":
            return f.size if f.size is not None else 0
        elif sort_by == "modified":
            return f.modified if f.modified is not None else 0
        elif sort_by == "type":
            return f.file_type.value
        else:
            return f.name.lower()

    return sorted(files, key=get_sort_key, reverse=descending)

In [ ]:
import time

# Test sort_files
now = time.time()
test_files = [
    FileInfo(name="zebra.txt", path="/zebra.txt", is_directory=False, size=100, modified=now-100),
    FileInfo(name="apple.txt", path="/apple.txt", is_directory=False, size=500, modified=now-50),
    FileInfo(name="mango.txt", path="/mango.txt", is_directory=False, size=200, modified=now),
]

# Sort by name (ascending)
result = sort_files(test_files, sort_by="name", descending=False)
assert [f.name for f in result] == ["apple.txt", "mango.txt", "zebra.txt"]

# Sort by name (descending)
result = sort_files(test_files, sort_by="name", descending=True)
assert [f.name for f in result] == ["zebra.txt", "mango.txt", "apple.txt"]

# Sort by size
result = sort_files(test_files, sort_by="size", descending=False)
assert [f.name for f in result] == ["zebra.txt", "mango.txt", "apple.txt"]

# Sort by modified (most recent first)
result = sort_files(test_files, sort_by="modified", descending=True)
assert result[0].name == "mango.txt"  # Most recently modified

print("sort_files tests passed!")

sort_files tests passed!


## Utility Functions

In [ ]:
#| export
def limit_files(
    files: List[FileInfo],       # Files to limit
    max_results: Optional[int]   # Maximum number of results (None = no limit)
) -> List[FileInfo]:  # Limited file list
    """Limit the number of files returned."""
    if max_results is None:
        return files
    return files[:max_results]

In [ ]:
# Test limit_files
test_files = [FileInfo(name=f"file{i}.txt", path=f"/file{i}.txt", is_directory=False) for i in range(10)]

assert len(limit_files(test_files, None)) == 10  # No limit
assert len(limit_files(test_files, 5)) == 5
assert len(limit_files(test_files, 20)) == 10  # Limit greater than list size
assert len(limit_files(test_files, 0)) == 0

print("limit_files tests passed!")

limit_files tests passed!


In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()